In [1]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
import evaluate
metric=evaluate.load("rouge")

In [3]:
#Data Handling
import numpy as np
import pandas as pd
from datasets import Dataset
import shutil

#Data Visualization
import plotly.express as px
import plotly.graph_objs as go
import plotly.subplots as sp
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import plotly.io as pio
from IPython.display import display
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

#Statitics and Mathematics
from scipy import stats
import statsmodels.api as sm
from scipy.stats import shapiro,skew,anderson,kstest,gaussian_kde,spearmanr
import math

In [4]:
import warnings
warnings.filterwarnings('ignore')


In [5]:
#transformers
from transformers import BartTokenizer,BartForConditionalGeneration
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import pipeline
from transformers import DataCollatorForSeq2Seq
import torch
import evaluate

#other nlp libraries
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
pd.set_option('display.max_colwidth',1000)

In [7]:
seed=42
colormap='cividis'
template='plotly_dark'

In [8]:
import torch

if torch.cuda.is_available():
    print("GPU is available. \nUsing GPU")
    device = torch.device('cuda')
else:
    print("GPU is not available. \nUsing CPU")
    device = torch.device('cpu')
#checking is available

GPU is not available. 
Using CPU


In [9]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.10.0+cpu
False
0


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.fc1 = nn.Linear(5, 10)
        self.fc2 = nn.Linear(10, 2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Now you can instantiate it
x = torch.rand(5, 5).to(device)
model = MyModel().to(device)


In [11]:
x=torch.rand(5,5).to(device)
model=MyModel().to(device)

In [12]:
output = model(x)   # runs on GPU if device is cuda


In [13]:
df1=pd.read_csv("samsum-test.csv")
df2=pd.read_csv("samsum-validation.csv")
df3=pd.read_csv("samsum-train.csv")
df=pd.concat([df1,df2,df3],ignore_index=True)

In [14]:
df.head(2)

,id,dialogue,summary
0,13862856,"Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye",Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
1,13729565,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric: I know! And shows how Americans see Russian ;)\r\nRob: And it's really funny!\r\nEric: I know! I especially like the train part!\r\nRob: Hahaha! No one talks to the machine like that!\r\nEric: Is this his only stand-up?\r\nRob: Idk. I'll check.\r\nEric: Sure.\r\nRob: Turns out no! There are some of his stand-ups on youtube.\r\nEric: Gr8! I'll watch them now!\r\nRob: Me too!\r\nEric: MACHINE!\r\nRob: MACHINE!\r\nEric: TTYL?\r\nRob: Sure :),Eric and Rob are going to watch a stand-up on youtube.


In [15]:
df.isnull().sum()

id          0
dialogue    1
summary     0
dtype: int64

In [16]:
df=df.dropna()

In [17]:
df = df.drop_duplicates()

In [18]:
from sklearn.model_selection import train_test_split

train_df,temp_df=train_test_split(df,test_size=0.2,random_state=42)
val_df,test_df=train_test_split(temp_df,test_size=0.5,random_state=42)

In [19]:
from datasets import Dataset
train_dataset=Dataset.from_pandas(train_df)
val_dataset=Dataset.from_pandas(val_df)
test_dataset=Dataset.from_pandas(test_df)

In [20]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("t5-small")

In [21]:
def preprocess_function(examples):
    inputs=["summarize:"+d for d in examples["dialogue"]]
    targets=examples["summary"]
    model_inputs=tokenizer(inputs,max_length=512,truncation=True,padding="max_length")
    labels=tokenizer(text_target=targets,max_length=128,truncation=True,padding="max_length")
    model_inputs["labels"]=labels["input_ids"]
    return model_inputs

In [22]:
train_dataset=train_dataset.map(preprocess_function,batched=True)

Map:   0%|          | 0/13094 [00:00<?, ? examples/s]

In [23]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [24]:
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [25]:
from transformers import TrainingArguments,Trainer

In [26]:
training_args=TrainingArguments(num_train_epochs=5)

In [27]:
rouge=evaluate.load("rouge")

In [28]:
import transformers
print(transformers.__version__)

5.5.4


In [29]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer)

In [30]:
trainer.train()

Step,Training Loss
500,0.807659
1000,0.473530
1500,0.465537
2000,0.442792
2500,0.442650
3000,0.444214
3500,0.425773
4000,0.430518
4500,0.426816
5000,0.417369


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8185, training_loss=0.45241202492501287, metrics={'train_runtime': 132114.0383, 'train_samples_per_second': 0.496, 'train_steps_per_second': 0.062, 'total_flos': 8860827742371840.0, 'train_loss': 0.45241202492501287, 'epoch': 5.0})

In [31]:
trainer.save_model("./summarization_model")
tokenizer.save_pretrained("./summarization_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./summarization_model\\tokenizer_config.json',
 './summarization_model\\tokenizer.json')

In [36]:
sample_text = test_df.iloc[0]["dialogue"]

inputs = tokenizer(
    "summarize: " + sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=100,
    num_beams=4
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original Dialogue:\n", sample_text)
print("\nGenerated Summary:\n", summary)

Original Dialogue:
 Finn: Hey Rory, how's it going?
Rory: Hi, man, not too bad, how are you and Jo coping?
Finn: I am feeling a bit stressed to be honest! I thought I was OK, but then started doing a few more things and got sucked into the wedding pressure!
Rory: Yes, I remember mine, Charlotte and me were so relieved when it was all over!
Finn: How's that speech coming along?
Rory: Well, not done a lot of it yet, made a few notes, embarrassing anecdotes etc.
Finn: Oh, shit! Don't be too candid with them, I don't want Jo to know all my sordid secrets!
Rory: Ooh, we yes, we go back a long way, I know about all your skeletons!
Finn: Keep it light and funny, and relatively clean, remember my nieces and nephews will be there!
Rory: Don't worry, mate!
Finn: How's your kilt fitting going, mine looks ace!
Rory: Hmm, let's just say, I've not really got the legs to pull it off, pale, hairless and skinny, not tanned and muscular!
Finn: Well, us Scotsman are not exactly renowned for our suntans! 